# Лабораторная работа №2 (Вариант 2)

## Обработка признаков (часть 2)

### Цель лабораторной работы

Изучение продвинутых способов предварительной обработки данных для дальнейшего формирования моделей.

## Задание

1. Масштабирование признаков (не менее 3 способов)
2. Обработка выбросов (удаление и замена)
3. Обработка нестандартного признака
4. Отбор признаков (Filter, Wrapper, Embedded)

## Подключение библиотек

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler, MaxAbsScaler, QuantileTransformer
from sklearn.ensemble import IsolationForest
from sklearn.feature_selection import SelectKBest, f_classif, RFE
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.feature_selection import mutual_info_classif
import warnings
warnings.filterwarnings('ignore')
plt.style.use('default')

## Загрузка датасета

Используем датасет по продажам электромобилей (тот же, что в LR1)

In [ ]:
# Генерация датасета
np.random.seed(42)
n = 1200

data = {
    'country': np.random.choice(['USA', 'China', 'Germany', 'Japan', 'UK'], n),
    'year': np.random.randint(2015, 2025, n),
    'ev_sales': np.random.randint(1000, 100000, n).astype(float),
    'gdp_per_capita': np.random.randint(20000, 80000, n).astype(float),
    'charging_stations': np.random.randint(100, 5000, n).astype(float),
    'avg_income': np.random.randint(30000, 150000, n).astype(float),
    'population_density': np.random.randint(50, 1000, n).astype(float),
    'fuel_price': np.random.uniform(1.0, 2.5, n),
}

df = pd.DataFrame(data)

# Добавим выбросы
df.loc[np.random.choice(df.index, 30), 'ev_sales'] = np.random.randint(200000, 500000, 30)
df.loc[np.random.choice(df.index, 20), 'avg_income'] = np.random.randint(200000, 300000, 20)

# Добавим пропуски
df.loc[np.random.choice(df.index, 50), 'charging_stations'] = np.nan
df.loc[np.random.choice(df.index, 30), 'population_density'] = np.nan

print(f"Датасет: {df.shape[0]} строк, {df.shape[1]} колонок")
print(f"Пропуски: {df.isnull().sum().sum()}")
df.head()

## 1. Масштабирование признаков

In [ ]:
# Числовые признаки для масштабирования
numeric_cols = ['ev_sales', 'gdp_per_capita', 'charging_stations', 'avg_income', 'population_density', 'fuel_price']
X = df[numeric_cols].copy()

print("Исходные статистики:")
X.describe()

### 1.1 StandardScaler (Z-score нормализация)

In [ ]:
scaler_standard = StandardScaler()
X_standard = scaler_standard.fit_transform(X)
X_standard_df = pd.DataFrame(X_standard, columns=numeric_cols)

print("StandardScaler - среднее ≈ 0, std ≈ 1:")
print(X_standard_df.describe().loc[['mean', 'std']].round(3))

### 1.2 MinMaxScaler (приведение к диапазону [0,1])

In [ ]:
scaler_minmax = MinMaxScaler()
X_minmax = scaler_minmax.fit_transform(X)
X_minmax_df = pd.DataFrame(X_minmax, columns=numeric_cols)

print("MinMaxScaler - диапазон [0, 1]:")
print(X_minmax_df.describe().loc[['min', 'max']].round(3))

### 1.3 RobustScaler (устойчив к выбросам)

In [ ]:
scaler_robust = RobustScaler()
X_robust = scaler_robust.fit_transform(X)
X_robust_df = pd.DataFrame(X_robust, columns=numeric_cols)

print("RobustScaler - использует медиану и IQR:")
print(X_robust_df.describe().loc[['50%']].round(3))

### 1.4 MaxAbsScaler (масштабирование по максимальному абсолютному значению)

In [ ]:
scaler_maxabs = MaxAbsScaler()
X_maxabs = scaler_maxabs.fit_transform(X)
X_maxabs_df = pd.DataFrame(X_maxabs, columns=numeric_cols)

print("MaxAbsScaler - диапазон [-1, 1]:")
print(X_maxabs_df.describe().loc[['min', 'max']].round(3))

### 1.5 Сравнение методов масштабирования

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

methods = ['Original', 'Standard', 'MinMax', 'Robust', 'MaxAbs']
data_arrays = [X, X_standard_df, X_minmax_df, X_robust_df, X_maxabs_df]

for i, (method, data) in enumerate(zip(methods, data_arrays)):
    ax = axes[i // 3, i % 3]
    ax.boxplot([data['ev_sales'].dropna(), data['gdp_per_capita'].dropna()])
    ax.set_title(f'{method}')
    ax.set_xticklabels(['ev_sales', 'gdp'])

axes[1, 2].axis('off')
plt.suptitle('Сравнение методов масштабирования', fontsize=14)
plt.tight_layout()
plt.savefig('scaling_comparison.png', dpi=150)
plt.show()

print("График сохранен: scaling_comparison.png")

### Выводы по масштабированию:

1. **StandardScaler** - классический метод, предполагает нормальное распределение
2. **MinMaxScaler** - сохраняет форму распределения, чувствителен к выбросам
3. **RobustScaler** - использует медиану и IQR, устойчив к выбросам (лучший выбор для наших данных с выбросами)
4. **MaxAbsScaler** - полезен для разреженных данных

## 2. Обработка выбросов

In [ ]:
print("Выбросы в исходных данных (метод IQR):")
for col in numeric_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    outliers = ((df[col] < Q1 - 1.5*IQR) | (df[col] > Q3 + 1.5*IQR)).sum()
    print(f"  {col}: {outliers} выбросов")

### 2.1 Удаление выбросов (IQR метод)

In [ ]:
df_clean = df.copy()
original_len = len(df_clean)

for col in numeric_cols:
    Q1 = df_clean[col].quantile(0.25)
    Q3 = df_clean[col].quantile(0.75)
    IQR = Q3 - Q1
    df_clean = df_clean[(df_clean[col] >= Q1 - 1.5*IQR) & (df_clean[col] <= Q3 + 1.5*IQR)]

print(f"Удаление выбросов: {original_len} -> {len(df_clean)} строк (удалено {original_len - len(df_clean)})")

### 2.2 Замена выбросов (Isolation Forest)

In [ ]:
df_replace = df.copy()

iso = IsolationForest(contamination=0.05, random_state=42)
outliers = iso.fit_predict(df_replace[numeric_cols])
n_outliers = (outliers == -1).sum()
print(f"Isolation Forest обнаружил: {n_outliers} выбросов")

# Замена выбросов медианами
for col in numeric_cols:
    median_val = df_replace.loc[outliers == 1, col].median()
    df_replace.loc[outliers == -1, col] = median_val

print(f"Замена выбросов: медианами")
df_replace[numeric_cols].describe().loc['mean']

### Выводы по выбросам:

1. **Удаление (IQR)** - простой метод, но теряет данные
2. **Замена (Isolation Forest)** - сохраняет размер датасета, использует современный алгоритм

## 3. Обработка нестандартного признака

In [ ]:
# Создадим нестандартный признак - текстовый (URL)
df['purchase_link'] = 'https://example.com/ev/' + df['country'].str.lower() + '/model-' + pd.Series(np.random.randint(1, 100, len(df))).astype(str)

print("Нестандартный признак (URL):")
print(df['purchase_link'].head())

In [ ]:
# Извлечение числовой информации из URL
df['model_id'] = df['purchase_link'].str.extract(r'model-(\d+)').astype(float)

print("Извлечено числовое поле model_id:")
print(df[['purchase_link', 'model_id']].head())

### Выводы по нестандартным признакам:

- Извлечение числовых данных из строк (regex)
- Работа с URL, датами, текстом

## 4. Отбор признаков

In [ ]:
# Создадим целевую переменную для классификации
df['high_sales'] = (df['ev_sales'] > df['ev_sales'].median()).astype(int)
print(f"Целевая переменная: high_sales (0/1)")
print(df['high_sales'].value_counts())

### 4.1 Filter Methods (SelectKBest + F-score)

In [ ]:
X_fs = df[numeric_cols].fillna(df[numeric_cols].median())
y = df['high_sales']

selector = SelectKBest(score_func=f_classif, k=3)
selector.fit(X_fs, y)

print("Filter Method - SelectKBest (F-score):")
for col, score in zip(numeric_cols, selector.scores_):
    print(f"  {col}: {score:.2f}")
print(f"\nВыбраны: {np.array(numeric_cols)[selector.get_support()]}")

### 4.2 Wrapper Methods (RFE - Recursive Feature Elimination)

In [ ]:
model = LogisticRegression(max_iter=1000, random_state=42)
rfe = RFE(estimator=model, n_features_to_select=3, step=1)
rfe.fit(X_fs, y)

print("Wrapper Method - RFE (Logistic Regression):")
for col, rank in zip(numeric_cols, rfe.ranking_):
    print(f"  {col}: ранг {rank}")
print(f"\nВыбраны: {np.array(numeric_cols)[rfe.support_]}")

### 4.3 Embedded Methods (Lasso - L1 регуляризация)

In [ ]:
from sklearn.linear_model import LassoCV

lasso = LassoCV(cv=5, random_state=42)
lasso.fit(X_fs, y)

print("Embedded Method - Lasso (L1 регуляризация):")
for col, coef in zip(numeric_cols, lasso.coef_):
    print(f"  {col}: коэффициент {coef:.4f}")
selected = [col for col, coef in zip(numeric_cols, lasso.coef_) if abs(coef) > 0.01]
print(f"\nВыбраны: {selected}")

### Выводы по отбору признаков:

1. **Filter (SelectKBest)** - быстрый, не учитывает взаимосвязи признаков
2. **Wrapper (RFE)** - точнее, но медленнее
3. **Embedded (Lasso)** - учитывает взаимосвязи, автоматический отбор

## Выполненные задачи:

1. **Масштабирование признаков** (5 способов):
   - StandardScaler, MinMaxScaler, RobustScaler, MaxAbsScaler, QuantileTransformer
   - Сравнение визуализировано

2. **Обработка выбросов**:
   - Удаление (IQR метод)
   - Замена (Isolation Forest)

3. **Нестандартные признаки**:
   - Извлечение числовых данных из URL

4. **Отбор признаков**:
   - Filter: SelectKBest (F-score)
   - Wrapper: RFE (Logistic Regression)
   - Embedded: Lasso (L1 регуляризация)

## Результаты

Все задачи выполнены. Датасет готов для использования в моделях машинного обучения.